1. Data Cleaning

In [10]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import sklearn as sk
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
import re

df = pd.read_csv("/Users/bianca/_BIANCA_/climbing-prediction/data/raw/climbharder_df.csv")

In [4]:
# print(df.count())
print("Columns with missing values:")
print("----------------------------")
for col in df:
    if df[col].count()<669:
        print(f"{col} | {df[col].count()} | {669 - df[col].count()} missing values")

df.iloc[1]

Columns with missing values:
----------------------------
Arm Span (cm) | 666 | 3 missing values
Max Weight hangboard 18mm edge - Half crimp (KG)  (10 seconds) (added weight only) | 273 | 396 missing values
Max Weight hangboard 18mm edge - open crimp (KG) (10 seconds)  (added weight only) | 230 | 439 missing values
Min Edge used (mm, +kg if weight added ) - Half Crimp (10 seconds) | 222 | 447 missing values
Min Edge used (mm, +kg if weight added) - Open crimp (10 seconds)  | 187 | 482 missing values
Endurance training  | 645 | 24 missing values
Other activities (ie yoga, cardio) | 328 | 341 missing values
Max pull up reps | 538 | 131 missing values
5 rep max weighted pull ups | 361 | 308 missing values
max push ups reps | 440 | 229 missing values
max L-sit time  | 289 | 380 missing values


Timestamp                                                                                                           29/01/2017 20:17:27
Sex                                                                                                                                Male
Height (cm)                                                                                                                         180
Weight (KG)                                                                                                                          81
Arm Span (cm)                                                                                                                       180
How long have you been climbing for?                                                                                      3 - 3.5 years
Where do you climb?                                                                                                Indoor Climbing only
Hardest V Grade ever climbed                    

In [ ]:
pd.set_option('future.no_silent_downcasting', True)

# Remove all columns with NaN (simplification purposes)
df2 = df.copy(deep=True)
cols_drop = [col for col in df2 if df2[col].isnull().values.any()]

# df2 = df2.drop(cols_drop, axis=1)
print(df2["Sex"].unique())
df2["Sex"] = df2["Sex"].replace({
    "Male":0,
    "Female":1
})

df["Height (cm)"] = df2["Height (cm)"].replace({
    
})

print(df2)

print(df2["Sex"].value_counts())

'''df2 = df2.select_dtypes(include='number')

print(list(df2.columns.values))

plt.figure(figsize=(15,10))

# Find correlation between numerical values - would not make much sense 
corrs = df2.corr()
sns.heatmap(round(corrs,2), cmap='RdBu', annot=True, 
            annot_kws={"size": 7}, vmin=-1, vmax=1)'''

# cols_str = [col for col in df2 if df2[col].dtype]

['Male' 'Female']
               Timestamp Sex Height (cm) Weight (KG) Arm Span (cm)  \
0    29/01/2017 20:12:46   0         173          77           178   
1    29/01/2017 20:17:27   0         180          81           180   
2    29/01/2017 20:28:14   0         178          67           175   
3    29/01/2017 20:51:08   0         173          70           178   
4    29/01/2017 21:03:19   0         184          84           197   
..                   ...  ..         ...         ...           ...   
664  24/01/2026 12:35:01   0      175.25        56.7           176   
665  17/03/2026 07:24:12   0         183          82           183   
666  19/03/2026 20:53:55   0      167.64      58.967        170.18   
667  07/04/2026 11:50:10   0       172.2        72.7           181   
668  12/06/2026 06:32:00   0         183          79           183   

    How long have you been climbing for?          Where do you climb?  \
0                          4.5 - 5 years  Indoor and outdoor climbin

array(['173', '180', '178', '184', '176', '186', '185', '175', '168',
       '157.5', '163', '174', '188', '164', '190', '177', '179', '185.4',
       '165.1', '157', '183', '167', '182', '170', '62', '193', '172.72',
       '195', '165', '182.8', '171', '160', '172', '187.96', '181',
       '1.68', '177.4', '191', '180.5', '153', '152', '194', '187', '154',
       '180.3', '192', '190.5', '198', '167cm', '189',
       '5 ft 8inches. Im amurican i dont know what centimeters are',
       '196', '159.38', '166', '182.88', '1.67', '159', '173 cm', '162.5',
       '172.7', '162', '177.8', '167.5', '155', '156', '169', '158',
       '182.9', '177.5', '1295', '175.26', '162.56', '167.64', '168.5',
       '185.3', '174.3', '110', '158.75', '201.1', '150', '8', '167.6',
       '1.75', '169.2', '161', '175.25', '172.2'], dtype=object)

In [ ]:
df2

,Timestamp,Sex,Height (cm),Weight (KG),Arm Span (cm),How long have you been climbing for?,Where do you climb?,Hardest V Grade ever climbed,Hardest V Grade climbed in the Last 3 months,The V grade you can send 90-100% of routes,...,Frequency of Endurance training sesions per week,Endurance training,General Strength Training frequency per week,Time spent General strength training (hours),Type of Strength training,"Other activities (ie yoga, cardio)",Max pull up reps,5 rep max weighted pull ups,max push ups reps,max L-sit time
0,29/01/2017 20:12:46,0,173,77,178,4.5 - 5 years,Indoor and outdoor climbing,V8,V8,V6,...,1,4x4,3,4,"Antagonists, Legs, Core",NaN,15,29kg,40,30
1,29/01/2017 20:17:27,0,180,81,180,3 - 3.5 years,Indoor Climbing only,V3,V3,V1,...,1,Laps of routes,2,2,"Antagonists, Legs, Core, Upper body pulling, U...","Yoga, stretching",11,5kg,24,15sec
2,29/01/2017 20:28:14,0,178,67,175,.5 - 1 years,Indoor and outdoor climbing,V7,V6,V5,...,2,"4x4, ARC, route climbing intervals",3,2,"Antagonists, Core, Upper body pulling, Upper b...",soccer,17,20 kg,NaN,NaN
3,29/01/2017 20:51:08,0,173,70,178,9 - 9.5 years,Indoor and outdoor climbing,V5,V4,V3,...,1,"Laps of routes, route climbing intervals",0,0,"Antagonists, Legs, Core, No other strength tra...",NaN,8,NaN,30,NaN
4,29/01/2017 21:03:19,0,184,84,197,6.5 - 7 years,Indoor and outdoor climbing,V10,V10,V7,...,2,"4x4, Max moves, threshold intervals",2,1,"Core, Upper body pushing",NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,24/01/2026 12:35:01,0,175.25,56.7,176,7 - 7.5 years,Indoor and outdoor climbing,V9,V9,V7,...,0,I don't train for endurance,2,2,"Antagonists, Legs, Core, Upper body pulling, U...",Surfing,+30,22.7,+20,NaN
665,17/03/2026 07:24:12,0,183,82,183,0 - .5 years,Indoor Climbing only,V6,V6,V4,...,0,I don't train for endurance,3,2,"Upper body pulling, Upper body pushing",NaN,22,30,50,NaN
666,19/03/2026 20:53:55,0,167.64,58.967,170.18,3.5 - 4 years,Indoor and outdoor climbing,V8,V7,V6,...,1,"4x4, hangboard repeater protocols",0,0,No other strength training,yoga,10,NaN,20,NaN
667,07/04/2026 11:50:10,0,172.2,72.7,181,More than 15 years,Indoor and outdoor climbing,V8,V8,V7,...,1,"4x4, Laps of routes, route climbing intervals",1,1,"Legs, Core, Upper body pulling, Upper body pus...",Occasional yoga/stretching.,17,20kg,25,33


In [ ]:
df2["Height (cm)"].unique()
df2["Weight (KG)"].unique()

df["Height (cm)"].value_counts(dropna=False)
'''
height:
3 have strings
units - cm, m, ft
'''

'''
standardize:
    lowercase
    remove all text
if contains ft
    convert ft -> cm, remove ft
if contains m
    convert m -> cm, remove m
if contains cm
    remove cm
if > max or < min
    replace with NA
if < meter max
    conver to cm
else 
    stay the same
'''

# df["Weight (KG)"].value_counts(dropna=False)

0      False
1      False
2      False
3      False
4      False
       ...  
664    False
665    False
666    False
667    False
668    False
Name: Sex, Length: 669, dtype: bool